# Logistic Regression for Grain Disease Recognition

## Data Preprocessing

In [1]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from tqdm import tqdm
import joblib

In [2]:
GRAIN_TYPES = ["maize", "rice"]
MAIZE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_HD", "7_IM"]
RICE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_UN", "7_IM"]

We need to turn the pixels of the image into a feature vector to feed into the training model. We decided to use the HSV histogram as the embedding function to create the feature vector.

In [3]:
from logistic_regression_lib import collect_embeddings

## Train Logistic Regression Model

Collect embeddings from training images and train a logistic regression classifier. For this, we combine the grain type and category in order to try and train a generalized model for both rice and maize.

We will also use both the train and validation sets for this as we will use K-fold cross validation and it will automatically segment the dataset into train and val sets.

In [34]:
# Collect training and test data
X_train_maize, y_train_maize = collect_embeddings("maize", "train")
X_train_rice, y_train_rice = collect_embeddings("rice", "train")
X_val_maize, y_val_maize = collect_embeddings("maize", "train")
X_val_rice, y_val_rice = collect_embeddings("rice", "train")

X_train = np.concatenate((X_train_maize, X_train_rice, X_val_maize, X_val_rice))
y_train = np.concatenate((y_train_maize, y_train_rice, y_val_maize, y_val_rice))
print(f"\nTraining data shape: {X_train.shape}")
print(f"Number of classes: {len(set(y_train))}")
print(f"Classes: {sorted(set(y_train))}")

Processing train images: 100%|██████████| 14535/14535 [00:16<00:00, 897.90it/s] 


Processing train images: 100%|██████████| 23685/23685 [00:30<00:00, 770.20it/s] 


Processing train images: 100%|██████████| 14535/14535 [00:16<00:00, 882.87it/s]


Processing train images: 100%|██████████| 23685/23685 [00:26<00:00, 890.38it/s] 



Training data shape: (76440, 512)
Number of classes: 16
Classes: [np.str_('maize_0_NOR'), np.str_('maize_1_F&S'), np.str_('maize_2_SD'), np.str_('maize_3_MY'), np.str_('maize_4_AP'), np.str_('maize_5_BN'), np.str_('maize_6_HD'), np.str_('maize_7_IM'), np.str_('rice_0_NOR'), np.str_('rice_1_F&S'), np.str_('rice_2_SD'), np.str_('rice_3_MY'), np.str_('rice_4_AP'), np.str_('rice_5_BN'), np.str_('rice_6_UN'), np.str_('rice_7_IM')]


In [35]:
# Collect test data
X_test_maize, y_test_maize = collect_embeddings("maize", "test")
X_test_rice, y_test_rice = collect_embeddings("rice", "test")
X_test = np.concatenate((X_test_maize, X_test_rice))
y_test = np.concatenate((y_test_maize, y_test_rice))

print(f"Test data shape: {X_test.shape}")
print(f"Number of test classes: {len(set(y_test))}")
print(f"Test classes: {sorted(set(y_test))}")

Processing test images: 100%|██████████| 1900/1900 [00:02<00:00, 872.26it/s]


Processing test images: 100%|██████████| 3100/3100 [00:03<00:00, 850.59it/s]

Test data shape: (5000, 512)
Number of test classes: 16
Test classes: [np.str_('maize_0_NOR'), np.str_('maize_1_F&S'), np.str_('maize_2_SD'), np.str_('maize_3_MY'), np.str_('maize_4_AP'), np.str_('maize_5_BN'), np.str_('maize_6_HD'), np.str_('maize_7_IM'), np.str_('rice_0_NOR'), np.str_('rice_1_F&S'), np.str_('rice_2_SD'), np.str_('rice_3_MY'), np.str_('rice_4_AP'), np.str_('rice_5_BN'), np.str_('rice_6_UN'), np.str_('rice_7_IM')]


And lets encode the labels to integers for the logistic regression to work

In [36]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

print(f"Label encoding: {dict(enumerate(label_encoder.classes_))}")

Label encoding: {0: np.str_('maize_0_NOR'), 1: np.str_('maize_1_F&S'), 2: np.str_('maize_2_SD'), 3: np.str_('maize_3_MY'), 4: np.str_('maize_4_AP'), 5: np.str_('maize_5_BN'), 6: np.str_('maize_6_HD'), 7: np.str_('maize_7_IM'), 8: np.str_('rice_0_NOR'), 9: np.str_('rice_1_F&S'), 10: np.str_('rice_2_SD'), 11: np.str_('rice_3_MY'), 12: np.str_('rice_4_AP'), 13: np.str_('rice_5_BN'), 14: np.str_('rice_6_UN'), 15: np.str_('rice_7_IM')}


In [37]:
# Encode test labels
y_test_encoded = label_encoder.transform(y_test)
print(f"\nTest set size: {X_test.shape[0]} samples")


Test set size: 5000 samples


## Hyperparameter Tuning with Grid Search

Now let's optimize the logistic regression model by finding the best hyperparameters using GridSearchCV. We'll tune:
- **C**: Regularization strength (lower = stronger regularization)
- **penalty**: Type of regularization (L1 or L2)
- **class_weight**: Whether to balance class weights
- **solver**: Algorithm to use for optimization


In [ ]:
from sklearn.model_selection import GridSearchCV
import pandas as pd

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2', 'l1'],
    'class_weight': [None, 'balanced'],
    'solver': ['liblinear', 'saga'],
}

print("Starting Grid Search...")
print(f"Total combinations to test: {len(param_grid['C']) * len(param_grid['penalty']) * len(param_grid['class_weight']) * len(param_grid['solver'])}")

# Create base model
base_lr = LogisticRegression(max_iter=2000, random_state=42)

# Grid search with cross-validation using F1-weighted scoring
grid_search = GridSearchCV(
    base_lr,
    param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='f1_weighted',  # Use F1 instead of accuracy for imbalanced data
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

grid_search.fit(X_train, y_train_encoded)

print("GRID SEARCH RESULTS")
print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best cross-validation F1-score: {grid_search.best_score_:.4f}")


Starting Grid Search...
Total combinations to test: 48
Fitting 5 folds for each of 48 candidates, totalling 240 fits


/home/xandreiathome/Work/class-code/stintsy/06_tips_and_tricks/.conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/xandreiathome/Work/class-code/stintsy/06_tips_and_tricks/.conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/xandreiathome/Work/class-code/stintsy/06_tips_and_tricks/.conda/lib/python3.11/site-pack

KeyboardInterrupt: 

In [ ]:
# Get the best model
best_lr_model = grid_search.best_estimator_

# Evaluate on test set
y_test_pred_best = best_lr_model.predict(X_test)

best_accuracy = best_lr_model.score(X_test, y_test_encoded)
best_f1 = f1_score(y_test_encoded, y_test_pred_best, average="weighted")
best_recall = recall_score(y_test_encoded, y_test_pred_best, average="weighted")
best_precision = precision_score(y_test_encoded, y_test_pred_best, average="weighted")

print("\nTEST SET PERFORMANCE - BEST MODEL")
print(f"Test Accuracy:  {best_accuracy:.4f}")
print(f"Test F1-Score:  {best_f1:.4f}")
print(f"Test Recall:    {best_recall:.4f}")
print(f"Test Precision: {best_precision:.4f}")

In [ ]:
# Visualize top results
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[['param_C', 'param_penalty', 'param_class_weight', 'param_solver', 'mean_test_score', 'std_test_score']]
results_df = results_df.sort_values('mean_test_score', ascending=False)

print("\nTOP 10 PARAMETER COMBINATIONS (by mean F1-score)")
print("="*60)
print(results_df.head(10).to_string(index=False))


In [ ]:
# Display per-class metrics for best model
from sklearn.metrics import classification_report


print("\nPER-CLASS METRICS - BEST MODEL")
print("="*60)
best_report = classification_report(
    y_test_encoded, 
    y_test_pred_best, 
    target_names=label_encoder.classes_, 
    digits=4
)
print(best_report)

# Save the best model
joblib.dump(best_lr_model, "models/logistic_regression_tuned.pkl")
print("\nBest tuned model saved to models/logistic_regression_tuned.pkl")
